# Capstone — Improve main-paper LTR with median labels + PARS

We compare **FCFS** vs **LTR (main-paper style / ProD-M)** vs **PARS (ours)** vs **Oracle**, with request priority.

**Before running**
1. Runtime → Change runtime type → GPU (T4)
2. Accept license: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Paste your HF token in the next cell
4. Run cells in order

Start with `--limit 50`. Use `--limit 100` for the final report.

In [ ]:
# 1) token + mount Drive (so results survive disconnects)
import os
from google.colab import drive

os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # <-- paste token
drive.mount("/content/drive")

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2) clone + install
import shutil, os

if os.path.exists("/content/pairwise-ltr-scheduler"):
    shutil.rmtree("/content/pairwise-ltr-scheduler")

!git clone https://github.com/anmolsaluja/pairwise-ltr-scheduler.git
%cd /content/pairwise-ltr-scheduler
!pip install -q -r requirements.txt
!python scripts/check_setup.py

In [ ]:
# 3) smoke test — 10 gsm8k prompts (~5-15 min)
!python scripts/generate_labels.py --dataset gsm8k --limit 10 --device cuda --llm-profile llama32

In [ ]:
# 4) full run — change --limit to 100 for the report
!python scripts/run_all.py --limit 50 --device cuda --llm-profile llama32

In [ ]:
# 5) report extras (ID/OOD + median vs single-sample ablation)
!python scripts/eval_ood.py --device cuda
!python scripts/ablation_labels.py --device cuda --epochs 3 --limit 50

In [ ]:
# 6) save results to Drive
!mkdir -p /content/drive/MyDrive/capstone_results
!cp -r checkpoints data/processed /content/drive/MyDrive/capstone_results/
print("saved to Drive:/capstone_results")

### If Colab disconnects mid-run

Remount Drive, clone again, copy checkpoints back, then:

```python
!cp -r /content/drive/MyDrive/capstone_results/checkpoints .
!cp -r /content/drive/MyDrive/capstone_results/processed data/ 2>/dev/null || \
  !cp -r /content/drive/MyDrive/capstone_results/data/processed data/
!python scripts/run_all.py --skip-train --device cuda
```

If only labels exist (no ranker yet):

```python
!python scripts/train_prod_m.py --device cuda
!python scripts/train_ranker.py --device cuda
!python scripts/evaluate.py --device cuda
```